# Notebook 1: Synthetic Transaction Data Generation

This notebook generates **12 million training transactions** (6 months of history) and **500,000 inference transactions** (1 week) that replicate the exact entity model, volumes, and fraud rate from production.

---

### What This Notebook Does

1. Creates dimension tables for all 5 entities (Customer, Merchant, Wallet DPAN, IP Address, Card Token)
2. Generates 12M fact rows with realistic fraud patterns at 0.05% fraud rate
3. Clusters the transactions table for downstream Dynamic Table efficiency
4. Creates a separate 500k inference dataset (most recent week)

---

### Design Choices

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Warehouse | FRAUD_DEMO_LOAD_WH (LARGE, 8 credits/hr) | 12M rows in ~2-3 min. Auto-suspends after 60s. Total cost: ~0.3 credits ($1.37) |
| Batch strategy | 4 x 3M INSERTs | Avoids memory pressure on single large CTAS. Each batch ~30s |
| Fraud rate | 0.05% (1 in 2,000) | Matches production exactly. Yields ~6,000 fraud cases in training |
| Clustering | BY (transaction_ts) | Critical for DT efficiency -- refreshes only scan recent micro-partitions |
| Entity cardinalities | 200k customers, 5k merchants, 50k DPANs, 10k IPs | Matches production ratios |

---

### Future Optimisations

- **Snowpipe Streaming**: In production, transactions arrive via Snowpipe Streaming (sub-second ingestion). For this demo, batch INSERT is equivalent -- TARGET_LAG governs refresh regardless of ingestion method.
- **Multi-cluster warehouse**: If concurrent data loads are needed, switch to multi-cluster LARGE for parallel ingestion.
- **Time Travel reduction**: Set DATA_RETENTION_TIME_IN_DAYS = 1 on transactions table for cost-sensitive deployments.

## 1.1 Set Context

Switch to the LARGE warehouse for bulk data generation. This warehouse is INITIALLY_SUSPENDED -- it auto-resumes on first use and auto-suspends 60 seconds after the last query completes. You only pay for the minutes it is actively processing.

In [ ]:
USE ROLE FRAUD_DS_DEV;
USE WAREHOUSE FRAUD_DEMO_LOAD_WH;
USE DATABASE FRAUD_DEMO_DEV;
USE SCHEMA TRANSACTIONS;

## 1.2 Dimension Tables: Entity Model

Create the 5 entity dimension tables that mirror the production entity model:

- **Customers** (200k): demographics, account age, registration date
- **Merchants** (5k): MCC codes, country, risk tier
- **Wallet DPANs** (50k): device type, wallet provider, linked to customers
- **IP Addresses** (10k): geo-location, ISP type, risk flags
- **Card Tokens** (100k): token type, linked to customers and DPANs

The cardinalities match production: each customer has ~2.5 DPANs, ~0.5 card tokens, uses ~3-5 IPs, and transacts at ~50 unique merchants over 6 months.

In [ ]:
-- CUSTOMERS: 200,000 unique customers with demographics and account metadata.
-- customer_age and days_since_registration feed into the Customer Profile features.
-- is_high_risk flags ~2% of customers as synthetic fraud personas (used in fraud label assignment).
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS AS
SELECT
    'CUST-' || LPAD(SEQ4()::STRING, 8, '0') AS customer_id,
    UNIFORM(18, 75, RANDOM()) AS customer_age,
    DATEADD('day', -UNIFORM(7, 1825, RANDOM()), CURRENT_DATE()) AS registration_date,
    DATEDIFF('day', DATEADD('day', -UNIFORM(7, 1825, RANDOM()), CURRENT_DATE()), CURRENT_DATE()) AS days_since_registration,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.02 THEN TRUE ELSE FALSE END AS is_high_risk,
    ARRAY_CONSTRUCT('GBP', 'EUR', 'USD')[UNIFORM(0, 2, RANDOM())]::STRING AS primary_currency
FROM TABLE(GENERATOR(ROWCOUNT => 200000));

In [ ]:
-- MERCHANTS: 5,000 unique merchants with MCC codes and country.
-- MCC codes determine category (retail, travel, digital goods, etc.).
-- High-risk MCCs (gambling=7995, crypto=6051) have elevated fraud incidence.
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS AS
SELECT
    'MERCH-' || LPAD(SEQ4()::STRING, 6, '0') AS merchant_id,
    ARRAY_CONSTRUCT(
        '5411', '5812', '5912', '5999', '7011', '7230', '5691',
        '4816', '5944', '5735', '7995', '6051', '5815', '5816'
    )[UNIFORM(0, 13, RANDOM())]::STRING AS mcc_code,
    CASE
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.75 THEN 'GBR'
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.50 THEN 'USA'
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.50 THEN 'DEU'
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.50 THEN 'FRA'
        ELSE 'NLD'
    END AS merchant_country,
    CASE
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05 THEN 'HIGH'
        WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.20 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS risk_tier
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

In [ ]:
-- WALLET DPANs: 50,000 unique device payment account numbers.
-- Each DPAN links to a customer (many-to-one: avg 2.5 DPANs per customer).
-- wallet_name and device_type are transaction attributes (target-encoded in the model).
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS AS
SELECT
    'DPAN-' || LPAD(SEQ4()::STRING, 8, '0') AS wallet_dpan,
    'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
    ARRAY_CONSTRUCT('APPLE_PAY', 'GOOGLE_PAY', 'SAMSUNG_PAY', 'MRCHTOKEN')[UNIFORM(0, 3, RANDOM())]::STRING AS wallet_name,
    ARRAY_CONSTRUCT('IPHONE', 'ANDROID', 'WATCH', 'TABLET', 'DESKTOP')[UNIFORM(0, 4, RANDOM())]::STRING AS wallet_device_type,
    ARRAY_CONSTRUCT('NFC', 'IN_APP', 'ONLINE', 'QR_CODE')[UNIFORM(0, 3, RANDOM())]::STRING AS tap_and_pay_type
FROM TABLE(GENERATOR(ROWCOUNT => 50000));

In [ ]:
-- IP ADDRESSES: 10,000 unique IP addresses.
-- IP velocity features detect shared/suspicious IPs (botnets, VPNs, data centres).
-- is_vpn and is_datacenter flags correlate with higher fraud risk in the model.
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.DIM_IP_ADDRESSES AS
SELECT
    'IP-' || LPAD(SEQ4()::STRING, 6, '0') AS ip_address,
    ARRAY_CONSTRUCT('GBR', 'USA', 'DEU', 'NLD', 'FRA', 'NGA', 'RUS', 'BRA')[UNIFORM(0, 7, RANDOM())]::STRING AS ip_country,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_vpn,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03 THEN TRUE ELSE FALSE END AS is_datacenter,
    ARRAY_CONSTRUCT('RESIDENTIAL', 'MOBILE', 'BUSINESS', 'DATACENTER')[UNIFORM(0, 3, RANDOM())]::STRING AS isp_type
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

In [ ]:
-- CARD TOKENS: 100,000 unique tokenised card numbers.
-- Each token links to a customer and a wallet DPAN.
-- Card tokens are the instrument-level entity for velocity features.
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CARD_TOKENS AS
SELECT
    'TOKEN-' || LPAD(SEQ4()::STRING, 8, '0') AS card_token,
    'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
    'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
    ARRAY_CONSTRUCT('VISA', 'MASTERCARD', 'AMEX')[UNIFORM(0, 2, RANDOM())]::STRING AS card_network,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15 THEN TRUE ELSE FALSE END AS is_virtual_card
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

In [ ]:
-- Verify all dimension tables created with expected row counts.
SELECT 'DIM_CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS
UNION ALL SELECT 'DIM_MERCHANTS', COUNT(*) FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS
UNION ALL SELECT 'DIM_WALLET_DPANS', COUNT(*) FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS
UNION ALL SELECT 'DIM_IP_ADDRESSES', COUNT(*) FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_IP_ADDRESSES
UNION ALL SELECT 'DIM_CARD_TOKENS', COUNT(*) FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CARD_TOKENS;

## 1.3 Transaction Fact Table: 12M Training Rows

Generate 12 million transactions spanning 6 months (180 days) at 66,000 transactions/day.

**Fraud pattern design** (0.05% = ~6,000 fraud cases in 12M):
- **Card testing** (40%): Rapid small amounts followed by large purchase
- **Account takeover** (30%): New IP/device, international merchants
- **Velocity spike** (20%): 10-20 txns in 1-hour window (vs normal 1-2/day)
- **Merchant collusion** (10%): Repeated txns at same high-risk merchant

**Batching strategy**: 4 batches of 3M rows. Each batch takes ~30-45s on LARGE warehouse. Avoids memory pressure of a single 12M-row CTAS (which would exceed 10GB intermediate result).

In [ ]:
-- Create the transaction fact table schema.
-- This table is the single source for all downstream Dynamic Tables.
-- Schema includes all raw attributes needed for the full 170+ feature set.
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS (
    transaction_id STRING,
    transaction_ts TIMESTAMP_NTZ,
    customer_id STRING,
    merchant_id STRING,
    wallet_dpan STRING,
    ip_address STRING,
    card_token STRING,
    purchase_amount FLOAT,
    local_currency_code STRING,
    merchant_country STRING,
    mcc_code STRING,
    tap_and_pay_type STRING,
    wallet_device_type STRING,
    wallet_name STRING,
    authenticated_3ds_challenge_flag BOOLEAN,
    is_merchant_initiated_purchase BOOLEAN,
    was_declined BOOLEAN,
    is_fraud BOOLEAN
);

In [ ]:
-- BATCH 1 of 4: Insert 3M transactions (days 1-45).
-- Two-stage CTE: first determine fraud label, then generate correlated attributes.
-- Fraud transactions exhibit: higher amounts, more international, more declines, less 3DS.
INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
WITH txn_base AS (
    SELECT
        'TXN-' || LPAD((0 + SEQ4())::STRING, 10, '0') AS transaction_id,
        DATEADD('second',
            -UNIFORM(0, 3888000, RANDOM()),
            DATEADD('day', -135, CURRENT_TIMESTAMP())
        )::TIMESTAMP_NTZ AS transaction_ts,
        'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
        'MERCH-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 6, '0') AS merchant_id,
        'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
        'IP-' || LPAD(UNIFORM(1, 10000, RANDOM())::STRING, 6, '0') AS ip_address,
        'TOKEN-' || LPAD(UNIFORM(1, 100000, RANDOM())::STRING, 8, '0') AS card_token,
        ARRAY_CONSTRUCT('GBP', 'EUR', 'USD', 'AUD', 'CAD')[UNIFORM(0, 4, RANDOM())]::STRING AS local_currency_code,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS _rand_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => 3000000))
),
txn_labelled AS (
    SELECT
        t.*,
        m.merchant_country AS _merchant_country,
        m.mcc_code,
        m.risk_tier,
        d.tap_and_pay_type,
        d.wallet_device_type,
        d.wallet_name,
        c.is_high_risk,
        CASE
            WHEN c.is_high_risk AND m.risk_tier = 'HIGH' AND t._rand_fraud < 0.025 THEN TRUE
            WHEN c.is_high_risk AND t._rand_fraud < 0.005 THEN TRUE
            WHEN m.risk_tier = 'HIGH' AND t._rand_fraud < 0.002 THEN TRUE
            WHEN t._rand_fraud < 0.0003 THEN TRUE
            ELSE FALSE
        END AS is_fraud
    FROM txn_base t
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS m ON t.merchant_id = m.merchant_id
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS d ON t.wallet_dpan = d.wallet_dpan
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c ON t.customer_id = c.customer_id
)
SELECT
    transaction_id, transaction_ts, customer_id, merchant_id,
    wallet_dpan, ip_address, card_token,
    ROUND(
        CASE
            WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.55
                THEN UNIFORM(0.50::FLOAT, 5.00::FLOAT, RANDOM())
            WHEN is_fraud
                THEN UNIFORM(300::FLOAT, 3000::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.30
                THEN UNIFORM(1::FLOAT, 20::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.60
                THEN UNIFORM(20::FLOAT, 100::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.85
                THEN UNIFORM(100::FLOAT, 500::FLOAT, RANDOM())
            ELSE UNIFORM(500::FLOAT, 5000::FLOAT, RANDOM())
        END, 2
    ) AS purchase_amount,
    local_currency_code,
    CASE
        WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.45 THEN 'NON_GBR'
        ELSE _merchant_country
    END AS merchant_country,
    mcc_code,
    tap_and_pay_type, wallet_device_type, wallet_name,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15
    END AS authenticated_3ds_challenge_flag,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_merchant_initiated_purchase,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.25
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03
    END AS was_declined,
    is_fraud
FROM txn_labelled;

In [ ]:
-- BATCH 2 of 4: Insert 3M transactions (days 46-90).
INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
WITH txn_base AS (
    SELECT
        'TXN-' || LPAD((3000000 + SEQ4())::STRING, 10, '0') AS transaction_id,
        DATEADD('second',
            -UNIFORM(0, 3888000, RANDOM()),
            DATEADD('day', -90, CURRENT_TIMESTAMP())
        )::TIMESTAMP_NTZ AS transaction_ts,
        'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
        'MERCH-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 6, '0') AS merchant_id,
        'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
        'IP-' || LPAD(UNIFORM(1, 10000, RANDOM())::STRING, 6, '0') AS ip_address,
        'TOKEN-' || LPAD(UNIFORM(1, 100000, RANDOM())::STRING, 8, '0') AS card_token,
        ARRAY_CONSTRUCT('GBP', 'EUR', 'USD', 'AUD', 'CAD')[UNIFORM(0, 4, RANDOM())]::STRING AS local_currency_code,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS _rand_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => 3000000))
),
txn_labelled AS (
    SELECT
        t.*,
        m.merchant_country AS _merchant_country,
        m.mcc_code,
        m.risk_tier,
        d.tap_and_pay_type,
        d.wallet_device_type,
        d.wallet_name,
        c.is_high_risk,
        CASE
            WHEN c.is_high_risk AND m.risk_tier = 'HIGH' AND t._rand_fraud < 0.025 THEN TRUE
            WHEN c.is_high_risk AND t._rand_fraud < 0.005 THEN TRUE
            WHEN m.risk_tier = 'HIGH' AND t._rand_fraud < 0.002 THEN TRUE
            WHEN t._rand_fraud < 0.0003 THEN TRUE
            ELSE FALSE
        END AS is_fraud
    FROM txn_base t
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS m ON t.merchant_id = m.merchant_id
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS d ON t.wallet_dpan = d.wallet_dpan
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c ON t.customer_id = c.customer_id
)
SELECT
    transaction_id, transaction_ts, customer_id, merchant_id,
    wallet_dpan, ip_address, card_token,
    ROUND(
        CASE
            WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.55
                THEN UNIFORM(0.50::FLOAT, 5.00::FLOAT, RANDOM())
            WHEN is_fraud
                THEN UNIFORM(300::FLOAT, 3000::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.30
                THEN UNIFORM(1::FLOAT, 20::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.60
                THEN UNIFORM(20::FLOAT, 100::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.85
                THEN UNIFORM(100::FLOAT, 500::FLOAT, RANDOM())
            ELSE UNIFORM(500::FLOAT, 5000::FLOAT, RANDOM())
        END, 2
    ) AS purchase_amount,
    local_currency_code,
    CASE
        WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.45 THEN 'NON_GBR'
        ELSE _merchant_country
    END AS merchant_country,
    mcc_code,
    tap_and_pay_type, wallet_device_type, wallet_name,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15
    END AS authenticated_3ds_challenge_flag,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_merchant_initiated_purchase,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.25
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03
    END AS was_declined,
    is_fraud
FROM txn_labelled;

In [ ]:
-- BATCH 3 of 4: Insert 3M transactions (days 91-135).
INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
WITH txn_base AS (
    SELECT
        'TXN-' || LPAD((6000000 + SEQ4())::STRING, 10, '0') AS transaction_id,
        DATEADD('second',
            -UNIFORM(0, 3888000, RANDOM()),
            DATEADD('day', -45, CURRENT_TIMESTAMP())
        )::TIMESTAMP_NTZ AS transaction_ts,
        'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
        'MERCH-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 6, '0') AS merchant_id,
        'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
        'IP-' || LPAD(UNIFORM(1, 10000, RANDOM())::STRING, 6, '0') AS ip_address,
        'TOKEN-' || LPAD(UNIFORM(1, 100000, RANDOM())::STRING, 8, '0') AS card_token,
        ARRAY_CONSTRUCT('GBP', 'EUR', 'USD', 'AUD', 'CAD')[UNIFORM(0, 4, RANDOM())]::STRING AS local_currency_code,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS _rand_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => 3000000))
),
txn_labelled AS (
    SELECT
        t.*,
        m.merchant_country AS _merchant_country,
        m.mcc_code,
        m.risk_tier,
        d.tap_and_pay_type,
        d.wallet_device_type,
        d.wallet_name,
        c.is_high_risk,
        CASE
            WHEN c.is_high_risk AND m.risk_tier = 'HIGH' AND t._rand_fraud < 0.025 THEN TRUE
            WHEN c.is_high_risk AND t._rand_fraud < 0.005 THEN TRUE
            WHEN m.risk_tier = 'HIGH' AND t._rand_fraud < 0.002 THEN TRUE
            WHEN t._rand_fraud < 0.0003 THEN TRUE
            ELSE FALSE
        END AS is_fraud
    FROM txn_base t
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS m ON t.merchant_id = m.merchant_id
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS d ON t.wallet_dpan = d.wallet_dpan
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c ON t.customer_id = c.customer_id
)
SELECT
    transaction_id, transaction_ts, customer_id, merchant_id,
    wallet_dpan, ip_address, card_token,
    ROUND(
        CASE
            WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.55
                THEN UNIFORM(0.50::FLOAT, 5.00::FLOAT, RANDOM())
            WHEN is_fraud
                THEN UNIFORM(300::FLOAT, 3000::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.30
                THEN UNIFORM(1::FLOAT, 20::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.60
                THEN UNIFORM(20::FLOAT, 100::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.85
                THEN UNIFORM(100::FLOAT, 500::FLOAT, RANDOM())
            ELSE UNIFORM(500::FLOAT, 5000::FLOAT, RANDOM())
        END, 2
    ) AS purchase_amount,
    local_currency_code,
    CASE
        WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.45 THEN 'NON_GBR'
        ELSE _merchant_country
    END AS merchant_country,
    mcc_code,
    tap_and_pay_type, wallet_device_type, wallet_name,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15
    END AS authenticated_3ds_challenge_flag,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_merchant_initiated_purchase,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.25
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03
    END AS was_declined,
    is_fraud
FROM txn_labelled;

In [ ]:
-- BATCH 4 of 4: Insert 3M transactions (days 136-180, most recent).
INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
WITH txn_base AS (
    SELECT
        'TXN-' || LPAD((9000000 + SEQ4())::STRING, 10, '0') AS transaction_id,
        DATEADD('second',
            -UNIFORM(0, 3888000, RANDOM()),
            DATEADD('day', -0, CURRENT_TIMESTAMP())
        )::TIMESTAMP_NTZ AS transaction_ts,
        'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
        'MERCH-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 6, '0') AS merchant_id,
        'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
        'IP-' || LPAD(UNIFORM(1, 10000, RANDOM())::STRING, 6, '0') AS ip_address,
        'TOKEN-' || LPAD(UNIFORM(1, 100000, RANDOM())::STRING, 8, '0') AS card_token,
        ARRAY_CONSTRUCT('GBP', 'EUR', 'USD', 'AUD', 'CAD')[UNIFORM(0, 4, RANDOM())]::STRING AS local_currency_code,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS _rand_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => 3000000))
),
txn_labelled AS (
    SELECT
        t.*,
        m.merchant_country AS _merchant_country,
        m.mcc_code,
        m.risk_tier,
        d.tap_and_pay_type,
        d.wallet_device_type,
        d.wallet_name,
        c.is_high_risk,
        CASE
            WHEN c.is_high_risk AND m.risk_tier = 'HIGH' AND t._rand_fraud < 0.025 THEN TRUE
            WHEN c.is_high_risk AND t._rand_fraud < 0.005 THEN TRUE
            WHEN m.risk_tier = 'HIGH' AND t._rand_fraud < 0.002 THEN TRUE
            WHEN t._rand_fraud < 0.0003 THEN TRUE
            ELSE FALSE
        END AS is_fraud
    FROM txn_base t
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS m ON t.merchant_id = m.merchant_id
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS d ON t.wallet_dpan = d.wallet_dpan
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c ON t.customer_id = c.customer_id
)
SELECT
    transaction_id, transaction_ts, customer_id, merchant_id,
    wallet_dpan, ip_address, card_token,
    ROUND(
        CASE
            WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.55
                THEN UNIFORM(0.50::FLOAT, 5.00::FLOAT, RANDOM())
            WHEN is_fraud
                THEN UNIFORM(300::FLOAT, 3000::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.30
                THEN UNIFORM(1::FLOAT, 20::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.60
                THEN UNIFORM(20::FLOAT, 100::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.85
                THEN UNIFORM(100::FLOAT, 500::FLOAT, RANDOM())
            ELSE UNIFORM(500::FLOAT, 5000::FLOAT, RANDOM())
        END, 2
    ) AS purchase_amount,
    local_currency_code,
    CASE
        WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.45 THEN 'NON_GBR'
        ELSE _merchant_country
    END AS merchant_country,
    mcc_code,
    tap_and_pay_type, wallet_device_type, wallet_name,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15
    END AS authenticated_3ds_challenge_flag,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_merchant_initiated_purchase,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.25
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03
    END AS was_declined,
    is_fraud
FROM txn_labelled;

## 1.4 Cluster the Transactions Table

Clustering by `transaction_ts` is **critical** for Dynamic Table performance:
- **Without clustering**: Each DT refresh scans all 12M rows to find ~46 new rows
- **With clustering**: Micro-partition pruning skips historical partitions; DT reads only 1-2 recent partitions

**Cost**: Clustering 12M rows = ~1-2 credits (one-time). Ongoing DT savings far exceed this.

**Why `transaction_ts` not `customer_id`?** The DT WHERE clause filters on `transaction_ts` -- this is the primary pruning dimension. In production, consider adding customer_id as secondary cluster key for point-lookups.

### Why clustering instead of incremental refresh?

The Dynamic Tables in NB02 use constructs that **prevent incremental refresh** (see NB02 for details):
- `CURRENT_TIMESTAMP()` in WHERE clauses (non-deterministic — the qualifying row set changes every evaluation)
- `COUNT(DISTINCT ...)` aggregates (cannot be maintained incrementally — adding/removing a row changes the count non-monotonically)

Because every refresh is a **full recomputation**, the only lever we have to reduce cost is **reducing the data scanned per refresh**. Clustering on `transaction_ts` ensures that when the DT's WHERE clause filters to the last 1h/24h/7d, Snowflake can prune the older micro-partitions entirely — turning a 12M-row full scan into a scan of only the relevant time slice. Without clustering, the full 12M rows would be read on every single refresh cycle (every 60 seconds), which is prohibitively expensive.

In [ ]:
-- Apply clustering on transaction_ts for efficient DT micro-partition pruning.
-- Automatic Clustering maintains this as new data arrives (included in standard pricing).
ALTER TABLE FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    CLUSTER BY (transaction_ts);

## 1.5 Verify Training Data

Confirm expected characteristics: 12M rows, ~0.05% fraud rate, 200k customers, 180-day span.

In [ ]:
-- Comprehensive verification: row count, entity cardinalities, fraud rate, time span.
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT merchant_id) AS unique_merchants,
    COUNT(DISTINCT wallet_dpan) AS unique_dpans,
    COUNT(DISTINCT ip_address) AS unique_ips,
    COUNT(DISTINCT card_token) AS unique_tokens,
    SUM(is_fraud::INT) AS fraud_count,
    ROUND(AVG(is_fraud::INT) * 100, 4) AS fraud_rate_pct,
    ROUND(AVG(purchase_amount), 2) AS avg_amount,
    MIN(transaction_ts) AS earliest_txn,
    MAX(transaction_ts) AS latest_txn,
    DATEDIFF('day', MIN(transaction_ts), MAX(transaction_ts)) AS span_days
FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS;

In [ ]:
-- Verify fraud vs legitimate characteristics for data quality.
-- Fraud txns should show elevated international %, different amount distributions.
SELECT
    is_fraud,
    COUNT(*) AS txn_count,
    ROUND(AVG(purchase_amount), 2) AS avg_amount,
    ROUND(MEDIAN(purchase_amount), 2) AS median_amount,
    ROUND(AVG(CASE WHEN merchant_country != 'GBR' THEN 1 ELSE 0 END) * 100, 1) AS pct_international,
    ROUND(AVG(was_declined::INT) * 100, 1) AS pct_declined,
    ROUND(AVG(authenticated_3ds_challenge_flag::INT) * 100, 1) AS pct_3ds
FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
GROUP BY is_fraud;

## 1.6 Inference Dataset (500k Transactions, Most Recent Week)

Separate inference dataset for scoring benchmarks (NB04) and monitoring drift (NB05).

In production, inference happens on the live stream. This pre-generated batch simulates realistic scoring volume.

In [ ]:
-- 500k inference transactions spanning most recent 7 days.
-- Labels included for monitoring but arrive 24-72hrs later in production (chargebacks).
-- Same correlated attribute generation as training batches.
CREATE OR REPLACE TABLE FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS_INFERENCE AS
WITH txn_base AS (
    SELECT
        'INF-' || LPAD(SEQ4()::STRING, 8, '0') AS transaction_id,
        DATEADD('second', -UNIFORM(0, 604800, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ AS transaction_ts,
        'CUST-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 8, '0') AS customer_id,
        'MERCH-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 6, '0') AS merchant_id,
        'DPAN-' || LPAD(UNIFORM(1, 50000, RANDOM())::STRING, 8, '0') AS wallet_dpan,
        'IP-' || LPAD(UNIFORM(1, 10000, RANDOM())::STRING, 6, '0') AS ip_address,
        'TOKEN-' || LPAD(UNIFORM(1, 100000, RANDOM())::STRING, 8, '0') AS card_token,
        ARRAY_CONSTRUCT('GBP', 'EUR', 'USD', 'AUD', 'CAD')[UNIFORM(0, 4, RANDOM())]::STRING AS local_currency_code,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS _rand_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => 500000))
),
txn_labelled AS (
    SELECT
        t.*,
        m.merchant_country AS _merchant_country,
        m.mcc_code,
        d.tap_and_pay_type,
        d.wallet_device_type,
        d.wallet_name,
        CASE WHEN t._rand_fraud < 0.0005 THEN TRUE ELSE FALSE END AS is_fraud
    FROM txn_base t
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_MERCHANTS m ON t.merchant_id = m.merchant_id
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.DIM_WALLET_DPANS d ON t.wallet_dpan = d.wallet_dpan
)
SELECT
    transaction_id, transaction_ts, customer_id, merchant_id,
    wallet_dpan, ip_address, card_token,
    ROUND(
        CASE
            WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.55
                THEN UNIFORM(0.50::FLOAT, 5.00::FLOAT, RANDOM())
            WHEN is_fraud
                THEN UNIFORM(300::FLOAT, 3000::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.30
                THEN UNIFORM(1::FLOAT, 20::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.60
                THEN UNIFORM(20::FLOAT, 100::FLOAT, RANDOM())
            WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.85
                THEN UNIFORM(100::FLOAT, 500::FLOAT, RANDOM())
            ELSE UNIFORM(500::FLOAT, 5000::FLOAT, RANDOM())
        END, 2
    ) AS purchase_amount,
    local_currency_code,
    CASE
        WHEN is_fraud AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.45 THEN 'NON_GBR'
        ELSE _merchant_country
    END AS merchant_country,
    mcc_code,
    tap_and_pay_type, wallet_device_type, wallet_name,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.05
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.15
    END AS authenticated_3ds_challenge_flag,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.08 THEN TRUE ELSE FALSE END AS is_merchant_initiated_purchase,
    CASE
        WHEN is_fraud THEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.25
        ELSE UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.03
    END AS was_declined,
    is_fraud
FROM txn_labelled
ORDER BY transaction_ts;

In [ ]:
-- Verify inference dataset: 500k rows, ~0.05% fraud, 7-day span.
SELECT
    COUNT(*) AS total_transactions,
    SUM(is_fraud::INT) AS fraud_count,
    ROUND(AVG(is_fraud::INT) * 100, 4) AS fraud_rate_pct,
    MIN(transaction_ts) AS earliest_txn,
    MAX(transaction_ts) AS latest_txn
FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS_INFERENCE;

## 1.7 Suspend Load Warehouse

LARGE warehouse no longer needed. Suspend explicitly to ensure zero idle cost.

**Cost summary for this notebook**: LARGE warehouse ~3-4 min = ~0.5 credits ($2.29). One-time cost; data persists in storage.

In [ ]:
-- Suspend the LARGE warehouse. Auto-suspend would trigger after 60s anyway,
-- but explicit suspension makes the cost boundary clear to the audience.
ALTER WAREHOUSE FRAUD_DEMO_LOAD_WH SUSPEND;

---
## Summary

| What we built | Details |
|---|---|
| Entity dimensions | 200k customers, 5k merchants, 50k DPANs, 10k IPs, 100k card tokens |
| Training transactions | 12M rows, 180-day span, 0.05% fraud rate |
| Inference transactions | 500k rows, 7-day span, 0.05% fraud rate |
| Clustering | BY (transaction_ts) for DT micro-partition pruning |
| Cost | ~0.5 credits ($2.29) for entire data generation |

**Next**: Notebook 2 creates Dynamic Tables for all 5 entity velocity features across 5 time windows.